In [5]:
"""
Module 1 — Data Preparation
Project: Automatic Road Network Extraction with Connectivity and Topology Refinement
Dataset: SpaceNet 5 (SN5) — AOI 8, Mumbai, India
Institution: VR Siddhartha Engineering College, Vijayawada

Fixes applied:
  1. stride = patch_size (no overlap) → reduces patch count ~4x
  2. Skip patches where mask has no road pixels → removes empty/useless patches
  3. Save as .npz compressed instead of .npy → ~4x smaller files
  4. Store all patches in TWO files (train.npz, val.npz) instead of 68k small files

Naming convention:
  Image : SN5_roads_train_AOI_8_Mumbai_PS-RGB_chip0.tif
  Label : SN5_roads_train_AOI_8_Mumbai_geojson_roads_speed_chip0.geojson
"""

import numpy as np
import cv2
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm


# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

CONFIG = {
    "image_dir":  "/home/jupyter-228w1a0529/Major/Dataset-1/PS-RGB",
    "label_dir":  "/home/jupyter-228w1a0529/Major/Dataset-1/geojson_roads_speed",
    "output_dir": "/home/jupyter-228w1a0529/Major/Dataset-1/processed",

    "patch_size":        512,
    "stride":            512,    # ← NO overlap (was 256). Eliminates duplicate patches.
    "road_width_px":     3,
    "min_road_px":       50,     # ← Skip patch if fewer than 50 road pixels (removes empty patches)

    "val_split":         0.30,
    "random_seed":       42,

    "mean": [0.485, 0.456, 0.406],
    "std":  [0.229, 0.224, 0.225],
}


# ─────────────────────────────────────────────────────────────────────────────
# HELPER — FIND MATCHING GEOJSON
# ─────────────────────────────────────────────────────────────────────────────

def find_label(img_path: Path, label_dir: Path):
    stem = img_path.stem
    candidate = label_dir / (stem.replace("PS-RGB", "geojson_roads_speed") + ".geojson")
    if candidate.exists():
        return candidate
    chip_id = stem.split("_")[-1]
    for f in label_dir.glob(f"*{chip_id}.geojson"):
        return f
    return None


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — LOAD PS-RGB GeoTIFF
# ─────────────────────────────────────────────────────────────────────────────

def load_geotiff(image_path: str):
    with rasterio.open(image_path) as src:
        image = src.read([1, 2, 3])
        image = np.transpose(image, (1, 2, 0))
        transform = src.transform
        crs = src.crs
    image = np.clip(image, 0, 255).astype(np.uint8)
    return image, transform, crs


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — RASTERIZE ROADS → BINARY MASK
# ─────────────────────────────────────────────────────────────────────────────

def rasterize_roads(geojson_path, image_shape, transform, road_width_px=3):
    H, W = image_shape
    mask = np.zeros((H, W), dtype=np.uint8)
    gdf = gpd.read_file(geojson_path)
    if gdf.empty:
        return mask
    for geom in gdf.geometry:
        if geom is None:
            continue
        if geom.geom_type == "LineString":
            pts = _geo_to_pixel(list(geom.coords), transform)
            cv2.polylines(mask, [pts], isClosed=False, color=1, thickness=road_width_px)
        elif geom.geom_type == "MultiLineString":
            for line in geom.geoms:
                pts = _geo_to_pixel(list(line.coords), transform)
                cv2.polylines(mask, [pts], isClosed=False, color=1, thickness=road_width_px)
    return mask


def _geo_to_pixel(coords, transform):
    inv = ~transform
    pts = []
    for (x, y) in coords:
        col, row = inv * (x, y)
        pts.append([int(col), int(row)])
    return np.array(pts, dtype=np.int32).reshape(-1, 1, 2)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — TILE INTO PATCHES (no overlap, skip empty)
# ─────────────────────────────────────────────────────────────────────────────

def extract_patches(image, mask, patch_size=512, stride=512, min_road_px=50):
    """
    Sliding window tiling.
    - stride = patch_size  → no overlap, ~4x fewer patches than stride=256
    - min_road_px          → skip patches with almost no road pixels
    """
    H, W, _ = image.shape
    img_patches, msk_patches = [], []
    skipped_empty = 0

    for row in range(0, H - patch_size + 1, stride):
        for col in range(0, W - patch_size + 1, stride):
            img_p  = image[row:row + patch_size, col:col + patch_size]
            mask_p = mask [row:row + patch_size, col:col + patch_size]

            # Skip patches with too few road pixels
            if mask_p.sum() < min_road_px:
                skipped_empty += 1
                continue

            img_patches.append(img_p)
            msk_patches.append(mask_p)

    return img_patches, msk_patches, skipped_empty


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — NORMALIZE
# ─────────────────────────────────────────────────────────────────────────────

def normalize_patch(patch, mean=CONFIG["mean"], std=CONFIG["std"]):
    p = patch.astype(np.float32) / 255.0
    return (p - np.array(mean, np.float32)) / np.array(std, np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — AUGMENTATION
# ─────────────────────────────────────────────────────────────────────────────

def augment(image_patch, mask_patch):
    aug_images, aug_masks = [image_patch], [mask_patch]
    aug_images.append(cv2.flip(image_patch, 1)); aug_masks.append(cv2.flip(mask_patch, 1))
    aug_images.append(cv2.flip(image_patch, 0)); aug_masks.append(cv2.flip(mask_patch, 0))
    for k in [1, 2, 3]:
        aug_images.append(np.rot90(image_patch, k=k))
        aug_masks.append( np.rot90(mask_patch,  k=k))
    return aug_images, aug_masks


# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — FULL PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline(config=CONFIG):
    image_dir  = Path(config["image_dir"])
    label_dir  = Path(config["label_dir"])
    output_dir = Path(config["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)

    image_files = sorted(image_dir.glob("*.tif"))
    print(f"[INFO] Found {len(image_files)} image(s)")

    all_images, all_masks = [], []
    matched = skipped_label = total_skipped_empty = 0

    for img_path in tqdm(image_files, desc="Processing tiles"):
        label_path = find_label(img_path, label_dir)
        if label_path is None:
            skipped_label += 1
            continue

        matched += 1
        image, transform, _ = load_geotiff(str(img_path))
        mask = rasterize_roads(str(label_path), image.shape[:2], transform,
                               config["road_width_px"])

        img_patches, msk_patches, n_skipped = extract_patches(
            image, mask,
            patch_size=config["patch_size"],
            stride=config["stride"],
            min_road_px=config["min_road_px"]
        )
        total_skipped_empty += n_skipped

        for img_p, msk_p in zip(img_patches, msk_patches):
            aug_imgs, aug_msks = augment(img_p, msk_p)
            all_images.extend(aug_imgs)
            all_masks.extend(aug_msks)

    print(f"\n[INFO] Matched: {matched} | No label: {skipped_label} | Empty patches skipped: {total_skipped_empty}")
    print(f"[INFO] Total patches (after augmentation): {len(all_images)}")

    if len(all_images) == 0:
        print("[ERROR] No patches. Check paths.")
        return

    # ── Estimate disk usage ──────────────────────────────────────────────────
    n = len(all_images)
    img_mb  = n * 512 * 512 * 3 * 4 / (1024**2)   # float32 images
    msk_mb  = n * 512 * 512 * 4     / (1024**2)    # float32 masks
    raw_gb  = (img_mb + msk_mb) / 1024
    est_gb  = raw_gb / 4                            # ~4x compression ratio
    print(f"[INFO] Estimated disk: raw={raw_gb:.1f} GB → compressed≈{est_gb:.1f} GB")

    # ── Split ────────────────────────────────────────────────────────────────
    indices = list(range(n))
    train_idx, val_idx = train_test_split(indices, test_size=config["val_split"],
                                          random_state=config["random_seed"])
    print(f"[INFO] Train: {len(train_idx)} | Val: {len(val_idx)}")

    # ── Normalize and stack ──────────────────────────────────────────────────
    print("[INFO] Normalizing and stacking train patches...")
    train_imgs, train_msks = _stack_split(all_images, all_masks, train_idx, config)

    print("[INFO] Saving train.npz (compressed)...")
    np.savez_compressed(output_dir / "train.npz", images=train_imgs, masks=train_msks)
    del train_imgs, train_msks   # free RAM

    print("[INFO] Normalizing and stacking val patches...")
    val_imgs, val_msks = _stack_split(all_images, all_masks, val_idx, config)

    print("[INFO] Saving val.npz (compressed)...")
    np.savez_compressed(output_dir / "val.npz", images=val_imgs, masks=val_msks)
    del val_imgs, val_msks

    # ── Report final sizes ───────────────────────────────────────────────────
    for fname in ["train.npz", "val.npz"]:
        fpath = output_dir / fname
        size_mb = fpath.stat().st_size / (1024**2)
        print(f"[INFO] {fname}: {size_mb:.1f} MB")

    print("[INFO] ✅ Data preparation complete.")
    print(f"[INFO] Output: {output_dir.resolve()}")


def _stack_split(all_images, all_masks, indices, config):
    """Normalize patches for given indices and return stacked arrays."""
    imgs, msks = [], []
    for idx in tqdm(indices, desc="  Stacking"):
        img_norm = normalize_patch(all_images[idx], config["mean"], config["std"])
        imgs.append(np.transpose(img_norm, (2, 0, 1)).astype(np.float32))  # (3,H,W)
        msks.append(all_masks[idx].astype(np.float32))                      # (H,W)
    return np.stack(imgs), np.stack(msks)  # (N,3,H,W), (N,H,W)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 — PYTORCH DATASET
# ─────────────────────────────────────────────────────────────────────────────

try:
    import torch
    from torch.utils.data import Dataset, DataLoader

    class RoadDataset(Dataset):
        """
        Loads from a single train.npz or val.npz file.
        Much faster than loading thousands of individual .npy files.
        """
        def __init__(self, npz_path: str):
            data = np.load(npz_path)
            self.images = data["images"]   # (N, 3, H, W) float32
            self.masks  = data["masks"]    # (N, H, W)    float32
            print(f"[INFO] Loaded {len(self.images)} patches from {npz_path}")

        def __len__(self):
            return len(self.images)

        def __getitem__(self, idx):
            return (
                torch.from_numpy(self.images[idx]),
                torch.from_numpy(self.masks[idx]).unsqueeze(0)  # (1, H, W)
            )

    def get_dataloaders(output_dir: str, batch_size: int = 8, num_workers: int = 4):
        output_dir = Path(output_dir)
        train_ds = RoadDataset(output_dir / "train.npz")
        val_ds   = RoadDataset(output_dir / "val.npz")
        train_loader = DataLoader(train_ds, batch_size=batch_size,
                                  shuffle=True,  num_workers=num_workers, pin_memory=True)
        val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                                  shuffle=False, num_workers=num_workers, pin_memory=True)
        print(f"[INFO] Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
        return train_loader, val_loader

except ImportError:
    print("[WARN] PyTorch not found.")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 8 — VISUALIZATION
# ─────────────────────────────────────────────────────────────────────────────

def visualize_sample(image_path: str, geojson_path: str, n_patches: int = 3):
    image, transform, _ = load_geotiff(image_path)
    mask = rasterize_roads(geojson_path, image.shape[:2], transform)

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(image);  axes[0].set_title("PS-RGB Image");    axes[0].axis("off")
    overlay = image.copy(); overlay[mask == 1] = [255, 0, 0]
    axes[1].imshow(overlay); axes[1].set_title("Road Overlay");   axes[1].axis("off")
    plt.tight_layout(); plt.savefig("sample_overview.png", dpi=150); plt.show()

    img_patches, msk_patches, _ = extract_patches(
        image, mask,
        patch_size=CONFIG["patch_size"],
        stride=CONFIG["stride"],
        min_road_px=CONFIG["min_road_px"]
    )
    idxs = np.random.choice(len(img_patches), min(n_patches, len(img_patches)), replace=False)
    fig, axes = plt.subplots(n_patches, 2, figsize=(10, n_patches * 5))
    for i, idx in enumerate(idxs):
        axes[i][0].imshow(img_patches[idx]);              axes[i][0].set_title(f"Patch {idx} — Image"); axes[i][0].axis("off")
        axes[i][1].imshow(msk_patches[idx], cmap="gray"); axes[i][1].set_title(f"Patch {idx} — Mask");  axes[i][1].axis("off")
    plt.tight_layout(); plt.savefig("sample_patches.png", dpi=150); plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    run_pipeline(CONFIG)

    # After pipeline completes, load DataLoaders:
    # train_loader, val_loader = get_dataloaders(CONFIG["output_dir"], batch_size=8)
    # images, masks = next(iter(train_loader))
    # print("Batch shapes:", images.shape, masks.shape)
    # Expected → torch.Size([8, 3, 512, 512])  torch.Size([8, 1, 512, 512])

[INFO] Found 1016 image(s)


Processing tiles: 100%|██████████| 1016/1016 [00:26<00:00, 37.64it/s]



[INFO] Matched: 1016 | No label: 0 | Empty patches skipped: 1858
[INFO] Total patches (after augmentation): 13236
[INFO] Estimated disk: raw=51.7 GB → compressed≈12.9 GB
[INFO] Train: 9265 | Val: 3971
[INFO] Normalizing and stacking train patches...


  Stacking: 100%|██████████| 9265/9265 [00:27<00:00, 331.33it/s]


[INFO] Saving train.npz (compressed)...
[INFO] Normalizing and stacking val patches...


  Stacking: 100%|██████████| 3971/3971 [00:15<00:00, 250.84it/s]


[INFO] Saving val.npz (compressed)...
[INFO] train.npz: 6937.9 MB
[INFO] val.npz: 2975.0 MB
[INFO] ✅ Data preparation complete.
[INFO] Output: /home/jupyter-228w1a0529/Major/Dataset-1/processed
